In [4]:
!pip install osmnx folium

import osmnx as ox
import networkx as nx
import folium

ox.settings.use_cache = True


tam = (10.762622, 106.660172)
G = ox.graph_from_point(tam, dist=2000, network_type='drive')


A = (10.761004, 106.668519)
B = (10.7704634, 106.6696456)

node_A = ox.distance.nearest_nodes(G, A[1], A[0])
node_B = ox.distance.nearest_nodes(G, B[1], B[0])


route1 = nx.shortest_path(G, node_A, node_B, weight='length')


def h(u, v):
    x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
    x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
    return ((x1-x2)**2 + (y1-y2)**2)**0.5

route2 = nx.astar_path(G, node_A, node_B, heuristic=h, weight='length')


def ve_duong(G, route, ban_do, mau):
    coords = []
    for u, v in zip(route[:-1], route[1:]):
        data = G.get_edge_data(u, v)[0]
        if 'geometry' in data:
            xs, ys = data['geometry'].xy
            for i in range(len(xs)):
                coords.append((ys[i], xs[i]))
        else:
            coords.append((G.nodes[u]['y'], G.nodes[u]['x']))
            coords.append((G.nodes[v]['y'], G.nodes[v]['x']))

    folium.PolyLine(coords, color=mau, weight=5).add_to(ban_do)

m = folium.Map(location=A, zoom_start=14)

ve_duong(G, route1, m, 'blue')  # Dijkstra
ve_duong(G, route2, m, 'red')   # A*


folium.Marker(A, popup="Điểm A").add_to(m)
folium.Marker(B, popup="Điểm B").add_to(m)

m.save("map.html")
m